# Seurat And AnnData UMAP Inventory

Standalone notebook for inspecting the source Seurat `.rds` objects, confirming the cached AnnData `.h5ad` conversions, and remaking UMAP plots from exported Seurat coordinates.

This notebook is intentionally not numbered as part of the Notebook 00/01/02 pipeline. It does not rerun Notebook 00, Notebook 01, or Seurat-to-AnnData conversion.

## Output Contract

Outputs are written under:

`$PROJECT_ROOT/results/seurat_anndata_umap_inventory/$SEURAT_INVENTORY_RUN_LABEL/`

The default run label is `seurat_anndata_umap_inventory_v1`. Tables, plots, logs, and the executed notebook are saved separately so repeated runs remain auditable.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidate_roots = [cwd, cwd.parent]
if len(cwd.parents) >= 2:
    candidate_roots.append(cwd.parents[1])

for root in candidate_roots:
    src = root / "python_notebooks" / "src"
    if src.exists():
        sys.path.insert(0, str(src))
        repo_root = root
        break
else:
    raise RuntimeError("Could not locate python_notebooks/src from the current notebook directory")

repo_root

In [ ]:
import json
import platform
import shutil
import subprocess
from datetime import datetime

import anndata as ad
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from mge_organoid_python import (
    cached_h5ad_path,
    default_studies,
    load_cached_anndatas,
    missing_cached_h5ads,
    resolve_project_root,
    validate_source_paths,
)

## Run Settings

These environment variables control the run:

- `PROJECT_ROOT`: runtime data root; defaults to the Great Lakes project path.
- `SEURAT_INVENTORY_RUN_LABEL`: output run label.
- `SEURAT_INVENTORY_STUDIES`: optional study filter, separated by `:`, `,`, or spaces.
- `SEURAT_INVENTORY_RUN_R`: whether to inspect source Seurat objects with R.
- `SEURAT_INVENTORY_OVERWRITE_R`: whether to rerun Seurat inspection if tables already exist.
- `SEURAT_INVENTORY_SAVE_PLOTS`: whether to save UMAP PNGs.
- `SEURAT_INVENTORY_SHOW_PLOTS`: whether to display plots inline in the executed notebook.

In [ ]:
def env_bool(name, default=False):
    raw = os.environ.get(name)
    if raw is None:
        return bool(default)
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_study_filter(raw):
    if raw is None or not raw.strip() or raw.strip().lower() == "all":
        return None
    tokens = raw.replace(",", ":").replace(" ", ":").split(":")
    return [token for token in tokens if token]


PROJECT_ROOT = resolve_project_root(os.environ.get("PROJECT_ROOT"))
RESULTS_DIRNAME = os.environ.get("SEURAT_INVENTORY_RESULTS_DIRNAME", "seurat_anndata_umap_inventory")
RUN_LABEL = os.environ.get("SEURAT_INVENTORY_RUN_LABEL", "seurat_anndata_umap_inventory_v1")
RUN_R_INVENTORY = env_bool("SEURAT_INVENTORY_RUN_R", True)
REQUIRE_R_INVENTORY_MARKERS = env_bool("SEURAT_INVENTORY_REQUIRE_R_MARKERS", False)
OVERWRITE_R_INVENTORY = env_bool("SEURAT_INVENTORY_OVERWRITE_R", False)
SAVE_PLOTS = env_bool("SEURAT_INVENTORY_SAVE_PLOTS", True)
SHOW_PLOTS = env_bool("SEURAT_INVENTORY_SHOW_PLOTS", False)
MAX_UMAP_COLOR_COLUMNS = int(os.environ.get("SEURAT_INVENTORY_MAX_UMAP_COLOR_COLUMNS", "10"))

RESULTS_ROOT = PROJECT_ROOT / "results" / RESULTS_DIRNAME
RUN_DIR = RESULTS_ROOT / RUN_LABEL
TABLE_DIR = RUN_DIR / "tables"
PLOT_DIR = RUN_DIR / "plots"
LOG_DIR = RUN_DIR / "logs"
SEURAT_TABLE_DIR = TABLE_DIR / "seurat_object_inventory"
ANNDATA_TABLE_DIR = TABLE_DIR / "anndata_inventory"
UMAP_PLOT_DIR = PLOT_DIR / "umaps"

for path in [RUN_DIR, TABLE_DIR, PLOT_DIR, LOG_DIR, SEURAT_TABLE_DIR, ANNDATA_TABLE_DIR, UMAP_PLOT_DIR, RESULTS_ROOT / "executed"]:
    path.mkdir(parents=True, exist_ok=True)

run_parameters = pd.DataFrame(
    [
        {"key": "timestamp", "value": datetime.now().isoformat(timespec="seconds")},
        {"key": "repo_root", "value": str(repo_root)},
        {"key": "project_root", "value": str(PROJECT_ROOT)},
        {"key": "results_dirname", "value": RESULTS_DIRNAME},
        {"key": "run_label", "value": RUN_LABEL},
        {"key": "run_dir", "value": str(RUN_DIR)},
        {"key": "run_r_inventory", "value": str(RUN_R_INVENTORY)},
        {"key": "require_r_inventory_markers", "value": str(REQUIRE_R_INVENTORY_MARKERS)},
        {"key": "overwrite_r_inventory", "value": str(OVERWRITE_R_INVENTORY)},
        {"key": "save_plots", "value": str(SAVE_PLOTS)},
        {"key": "show_plots", "value": str(SHOW_PLOTS)},
        {"key": "max_umap_color_columns", "value": str(MAX_UMAP_COLOR_COLUMNS)},
    ]
)
run_parameters.to_csv(TABLE_DIR / "seurat_anndata_umap_inventory_run_parameters.tsv", sep="\t", index=False)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("TABLE_DIR:", TABLE_DIR)
print("PLOT_DIR:", PLOT_DIR)

## Runtime Diagnostics

The R-side Seurat inventory loads full `.rds` objects. Run that part on a compute node through Slurm, not on a login node.

In [ ]:
diagnostics = pd.DataFrame(
    [
        {"key": "hostname", "value": platform.node()},
        {"key": "python", "value": sys.executable},
        {"key": "python_version", "value": sys.version.replace("\n", " ")},
        {"key": "PROJECT_ROOT_env", "value": os.environ.get("PROJECT_ROOT", "")},
        {"key": "CONDA_DEFAULT_ENV", "value": os.environ.get("CONDA_DEFAULT_ENV", "")},
        {"key": "SLURM_JOB_ID", "value": os.environ.get("SLURM_JOB_ID", "")},
        {"key": "Rscript", "value": shutil.which("Rscript") or ""},
        {"key": "jupyter", "value": shutil.which("jupyter") or ""},
    ]
)
diagnostics.to_csv(TABLE_DIR / "seurat_anndata_umap_inventory_runtime_diagnostics.tsv", sep="\t", index=False)
display(diagnostics)

if platform.node().startswith("gl-login") and RUN_R_INVENTORY:
    print("WARNING: R Seurat inventory is enabled on a login node. Use Slurm for the full run.")

## Confirm Inputs

These are the canonical Seurat source objects and cached AnnData files. The notebook reads the cached `.h5ad` files and only inspects the `.rds` objects for object inventory.

In [ ]:
studies_all = default_studies()
study_filter = parse_study_filter(os.environ.get("SEURAT_INVENTORY_STUDIES"))
if study_filter is None:
    studies = studies_all
else:
    by_id = {study.study_id: study for study in studies_all}
    missing_ids = [study_id for study_id in study_filter if study_id not in by_id]
    if missing_ids:
        raise ValueError(f"Unknown study IDs in SEURAT_INVENTORY_STUDIES: {missing_ids}")
    studies = [by_id[study_id] for study_id in study_filter]

source_missing = validate_source_paths(studies)
cache_missing = missing_cached_h5ads(studies, project_root=PROJECT_ROOT)
if source_missing:
    raise FileNotFoundError(f"Missing source Seurat files: {source_missing}")
if cache_missing:
    raise FileNotFoundError(f"Missing cached H5AD files: {cache_missing}")

input_rows = []
for study in studies:
    seurat_path = Path(study.seurat_path).expanduser()
    h5ad_path = cached_h5ad_path(study, project_root=PROJECT_ROOT)
    input_rows.append(
        {
            "study_id": study.study_id,
            "label": study.label,
            "seurat_path": str(seurat_path),
            "seurat_size_gb": round(seurat_path.stat().st_size / 1024**3, 3),
            "assay": study.assay,
            "reduction": study.reduction,
            "expression_layer": study.expression_layer,
            "h5ad_path": str(h5ad_path),
            "h5ad_size_gb": round(h5ad_path.stat().st_size / 1024**3, 3),
        }
    )
input_paths_df = pd.DataFrame(input_rows)
input_paths_df.to_csv(TABLE_DIR / "seurat_anndata_umap_inventory_input_paths.tsv", sep="\t", index=False)
display(input_paths_df)

## Inspect Source Seurat Objects

The R helper writes object-level summaries without converting or modifying the source `.rds` files.

In [ ]:
rscript = shutil.which("Rscript")
r_inventory_script = repo_root / "python_notebooks" / "scripts" / "inspect_seurat_object.R"
if RUN_R_INVENTORY and rscript is None:
    raise RuntimeError("Rscript was not found. Run this through the mge-organoid-python Slurm environment.")
if RUN_R_INVENTORY and not r_inventory_script.exists():
    raise FileNotFoundError(f"Missing R inventory helper: {r_inventory_script}")

r_run_rows = []
for study in studies:
    outdir = SEURAT_TABLE_DIR / study.study_id
    outdir.mkdir(parents=True, exist_ok=True)
    expected = outdir / "seurat_inventory_complete.tsv"
    log_path = LOG_DIR / f"inspect_seurat_object.{study.study_id}.log"
    if expected.exists() and not OVERWRITE_R_INVENTORY:
        status = "skipped_existing"
        print(f"[{study.study_id}] using existing Seurat inventory: {outdir}")
    elif RUN_R_INVENTORY:
        cmd = [
            rscript,
            str(r_inventory_script),
            "--study_id",
            study.study_id,
            "--label",
            study.label,
            "--seurat",
            str(Path(study.seurat_path).expanduser()),
            "--h5ad",
            str(cached_h5ad_path(study, project_root=PROJECT_ROOT)),
            "--outdir",
            str(outdir),
        ]
        print(f"[{study.study_id}] running Seurat inventory")
        proc = subprocess.run(cmd, cwd=str(repo_root), text=True, capture_output=True)
        log_path.write_text(proc.stdout + "\n" + proc.stderr)
        if proc.returncode != 0:
            print(proc.stdout)
            print(proc.stderr)
            raise RuntimeError(f"Seurat inventory failed for {study.study_id}; see {log_path}")
        status = "ran"
    elif REQUIRE_R_INVENTORY_MARKERS:
        raise FileNotFoundError(
            f"Missing required Seurat inventory completion marker for {study.study_id}: {expected}. "
            "Run the Seurat inventory array first or set SEURAT_INVENTORY_RUN_R=1."
        )
    else:
        status = "disabled"
        print(f"[{study.study_id}] R inventory disabled")
    r_run_rows.append(
        {
            "study_id": study.study_id,
            "status": status,
            "inventory_dir": str(outdir),
            "completion_marker_path": str(expected),
            "log_path": str(log_path),
            "completion_marker_exists": expected.exists(),
        }
    )

r_inventory_run_df = pd.DataFrame(r_run_rows)
r_inventory_run_df.to_csv(TABLE_DIR / "seurat_inventory_run_manifest.tsv", sep="\t", index=False)
display(r_inventory_run_df)

## Seurat Inventory Tables

These tables answer what is inside each source Seurat object: assays, reductions, graphs, metadata, command history, active identities, misc/tool slots, and variable features where present.

In [ ]:
def read_optional_tsv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path, sep="\t")
    return pd.DataFrame()

seurat_file_rows = []
combined_tables = {}
for study in studies:
    study_dir = SEURAT_TABLE_DIR / study.study_id
    for path in sorted(study_dir.glob("*.tsv")):
        seurat_file_rows.append(
            {
                "study_id": study.study_id,
                "file": path.name,
                "path": str(path),
                "size_bytes": path.stat().st_size,
            }
        )
        key = path.name.replace(".tsv", "")
        if key.startswith("seurat_variable_features_"):
            key = "seurat_variable_features"
        df = read_optional_tsv(path)
        if not df.empty:
            df = df.copy()
            if "study_id" in df.columns:
                df["study_id"] = study.study_id
            else:
                df.insert(0, "study_id", study.study_id)
            combined_tables.setdefault(key, []).append(df)

seurat_file_manifest = pd.DataFrame(seurat_file_rows)
seurat_file_manifest.to_csv(TABLE_DIR / "seurat_inventory_file_manifest.tsv", sep="\t", index=False)

for key, dfs in combined_tables.items():
    out_df = pd.concat(dfs, ignore_index=True)
    out_df.to_csv(TABLE_DIR / f"combined_{key}.tsv", sep="\t", index=False)

object_summary = pd.concat(combined_tables.get("seurat_object_summary", []), ignore_index=True) if "seurat_object_summary" in combined_tables else pd.DataFrame()
metadata_columns = pd.concat(combined_tables.get("seurat_metadata_columns", []), ignore_index=True) if "seurat_metadata_columns" in combined_tables else pd.DataFrame()
reductions = pd.concat(combined_tables.get("seurat_reductions", []), ignore_index=True) if "seurat_reductions" in combined_tables else pd.DataFrame()
assays = pd.concat(combined_tables.get("seurat_assays", []), ignore_index=True) if "seurat_assays" in combined_tables else pd.DataFrame()

print("Seurat inventory files:", len(seurat_file_manifest))
if not object_summary.empty:
    display(object_summary.pivot(index="key", columns="study_id", values="value"))
if not assays.empty:
    display(assays)
if not reductions.empty:
    display(reductions)
if not metadata_columns.empty:
    display(metadata_columns.groupby("study_id").agg(n_metadata_columns=("column", "nunique")))

## Load Cached AnnData Files

The `.h5ad` files are opened with `backed="r"` so expression matrices stay on disk. Metadata and UMAP coordinates are still available for inspection and plotting.

In [ ]:
adatas, reports = load_cached_anndatas(studies, project_root=PROJECT_ROOT, backed="r")
reports_df = pd.DataFrame([report.as_dict() for report in reports])
reports_df.to_csv(ANNDATA_TABLE_DIR / "anndata_load_reports.tsv", sep="\t", index=False)
display(reports_df)

## AnnData Inventory

These summaries document what the current converter preserved in the cached AnnData objects.

In [ ]:
def preview_value(value, max_len=160):
    text = repr(value)
    text = text.replace("\n", " ").replace("\t", " ")
    if len(text) > max_len:
        return text[: max_len - 3] + "..."
    return text


def anndata_object_inventory(adatas):
    rows = []
    for study_id, adata_obj in adatas.items():
        rows.append(
            {
                "study_id": study_id,
                "n_obs": adata_obj.n_obs,
                "n_vars": adata_obj.n_vars,
                "n_obs_columns": adata_obj.obs.shape[1],
                "n_var_columns": adata_obj.var.shape[1],
                "layers": "; ".join(adata_obj.layers.keys()),
                "obsm": "; ".join(adata_obj.obsm.keys()),
                "varm": "; ".join(adata_obj.varm.keys()),
                "obsp": "; ".join(adata_obj.obsp.keys()),
                "uns": "; ".join(adata_obj.uns.keys()),
                "has_counts_layer": "counts" in adata_obj.layers,
                "has_X_umap": "X_umap" in adata_obj.obsm,
                "has_X_umap_seurat": "X_umap_seurat" in adata_obj.obsm,
            }
        )
    return pd.DataFrame(rows)


def axis_column_inventory(adatas, axis):
    attr = "obs" if axis == "obs" else "var"
    all_columns = sorted({column for adata_obj in adatas.values() for column in getattr(adata_obj, attr).columns})
    rows = []
    for column in all_columns:
        row = {"axis": axis, "column": column}
        for study_id, adata_obj in adatas.items():
            frame = getattr(adata_obj, attr)
            row[study_id] = column in frame.columns
            if column in frame.columns:
                row[f"{study_id}_dtype"] = str(frame[column].dtype)
        rows.append(row)
    return pd.DataFrame(rows)


def anndata_key_inventory(adatas):
    rows = []
    for study_id, adata_obj in adatas.items():
        for container_name in ["layers", "obsm", "varm", "obsp"]:
            container = getattr(adata_obj, container_name)
            for key in container.keys():
                value = container[key]
                rows.append(
                    {
                        "study_id": study_id,
                        "container": container_name,
                        "key": key,
                        "class": type(value).__name__,
                        "shape": "x".join(map(str, getattr(value, "shape", ""))) if hasattr(value, "shape") else "",
                    }
                )
        for key, value in adata_obj.uns.items():
            rows.append(
                {
                    "study_id": study_id,
                    "container": "uns",
                    "key": key,
                    "class": type(value).__name__,
                    "shape": "",
                    "preview": preview_value(value),
                }
            )
    return pd.DataFrame(rows)


def categorical_obs_summary(adata_obj, max_unique=50):
    rows = []
    for column in adata_obj.obs.columns:
        series = adata_obj.obs[column]
        nunique = int(series.nunique(dropna=True))
        if str(series.dtype) == "category" or nunique <= max_unique:
            counts = series.astype("string").fillna("<NA>").value_counts(dropna=False).head(15)
            rows.append(
                {
                    "column": column,
                    "dtype": str(series.dtype),
                    "n_unique": nunique,
                    "top_values": "; ".join(f"{idx}: {value}" for idx, value in counts.items()),
                }
            )
    return pd.DataFrame(rows)


def numeric_obs_summary(adata_obj):
    rows = []
    for column in adata_obj.obs.columns:
        values = pd.to_numeric(adata_obj.obs[column], errors="coerce")
        if values.notna().sum() == 0:
            continue
        rows.append(
            {
                "column": column,
                "n_non_missing": int(values.notna().sum()),
                "mean": float(values.mean()),
                "median": float(values.median()),
                "min": float(values.min()),
                "max": float(values.max()),
            }
        )
    return pd.DataFrame(rows)


anndata_inventory_df = anndata_object_inventory(adatas)
obs_columns_df = axis_column_inventory(adatas, "obs")
var_columns_df = axis_column_inventory(adatas, "var")
anndata_keys_df = anndata_key_inventory(adatas)

anndata_inventory_df.to_csv(ANNDATA_TABLE_DIR / "anndata_object_inventory.tsv", sep="\t", index=False)
obs_columns_df.to_csv(ANNDATA_TABLE_DIR / "anndata_obs_column_inventory.tsv", sep="\t", index=False)
var_columns_df.to_csv(ANNDATA_TABLE_DIR / "anndata_var_column_inventory.tsv", sep="\t", index=False)
anndata_keys_df.to_csv(ANNDATA_TABLE_DIR / "anndata_key_inventory.tsv", sep="\t", index=False)

cat_summaries = []
num_summaries = []
for study_id, adata_obj in adatas.items():
    cat_df = categorical_obs_summary(adata_obj)
    if not cat_df.empty:
        cat_df.insert(0, "study_id", study_id)
        cat_summaries.append(cat_df)
    num_df = numeric_obs_summary(adata_obj)
    if not num_df.empty:
        num_df.insert(0, "study_id", study_id)
        num_summaries.append(num_df)

categorical_obs_df = pd.concat(cat_summaries, ignore_index=True) if cat_summaries else pd.DataFrame()
numeric_obs_df = pd.concat(num_summaries, ignore_index=True) if num_summaries else pd.DataFrame()
categorical_obs_df.to_csv(ANNDATA_TABLE_DIR / "anndata_categorical_obs_summary.tsv", sep="\t", index=False)
numeric_obs_df.to_csv(ANNDATA_TABLE_DIR / "anndata_numeric_obs_summary.tsv", sep="\t", index=False)

display(anndata_inventory_df)
display(obs_columns_df)
display(var_columns_df)

## Conversion Scope Summary

The current converter preserves expression, counts, Seurat metadata, and the configured Seurat UMAP. Other Seurat slots are inspected here but not automatically transferred to AnnData.

In [ ]:
manifest_rows = []
conversion_scope_rows = []
for study_id, adata_obj in adatas.items():
    manifest = adata_obj.uns.get("conversion_manifest", {})
    if isinstance(manifest, dict):
        for key, value in manifest.items():
            manifest_rows.append({"study_id": study_id, "key": key, "value": value})
    conversion_scope_rows.append(
        {
            "study_id": study_id,
            "source_seurat_path_in_uns": adata_obj.uns.get("source_seurat_path", ""),
            "seurat_assay_in_uns": adata_obj.uns.get("seurat_assay", ""),
            "seurat_reduction_in_uns": adata_obj.uns.get("seurat_reduction", ""),
            "X_meaning": "Seurat data-like layer requested by StudySpec.expression_layer",
            "counts_layer_present": "counts" in adata_obj.layers,
            "counts_layer_meaning": "Seurat counts layer for the converted assay",
            "umap_keys_present": "; ".join([key for key in ["X_umap_seurat", "X_umap"] if key in adata_obj.obsm]),
            "obs_meaning": "Seurat obj@meta.data plus cell_id",
            "var_meaning": "feature_id from converted assay feature names",
            "not_transferred_by_converter": "other reductions; graphs; neighbors; command history; misc/tools; images; scale.data unless explicitly chosen as data layer; alternative assays",
        }
    )

conversion_manifest_df = pd.DataFrame(manifest_rows)
conversion_scope_df = pd.DataFrame(conversion_scope_rows)
conversion_manifest_df.to_csv(ANNDATA_TABLE_DIR / "anndata_conversion_manifest.tsv", sep="\t", index=False)
conversion_scope_df.to_csv(TABLE_DIR / "seurat_to_anndata_conversion_scope.tsv", sep="\t", index=False)

display(conversion_manifest_df)
display(conversion_scope_df)

## Remake UMAPs From Exported Seurat Coordinates

These plots replot existing Seurat UMAP coordinates stored in `.obsm["X_umap_seurat"]`. They do not recompute PCA, neighbors, or UMAP.

In [ ]:
def sanitize_filename(value):
    allowed = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789._-"
    return "".join(ch if ch in allowed else "_" for ch in str(value))


def is_numeric_column(series):
    values = pd.to_numeric(series, errors="coerce")
    return values.notna().sum() > 0 and values.notna().sum() >= max(10, int(0.5 * len(series)))


def select_umap_color_columns(adata_obj, max_columns=10):
    priority = [
        "seurat_clusters",
        "SCT_snn_res.0.8",
        "RNA_snn_res.0.8",
        "orig.ident",
        "sample",
        "Sample",
        "sample_id",
        "condition",
        "Condition",
        "DIV",
        "cluster",
        "Cluster",
        "celltype",
        "cell_type",
        "cell.types",
        "predicted_cell_type",
        "nCount_RNA",
        "nFeature_RNA",
        "percent.mt",
    ]
    selected = []
    for column in priority:
        if column in adata_obj.obs.columns and column not in selected:
            selected.append(column)
    for column in adata_obj.obs.columns:
        if column in selected:
            continue
        nunique = int(adata_obj.obs[column].nunique(dropna=True))
        if 1 < nunique <= 30:
            selected.append(column)
        if len(selected) >= max_columns:
            break
    for column in adata_obj.obs.columns:
        if column in selected:
            continue
        if is_numeric_column(adata_obj.obs[column]):
            selected.append(column)
        if len(selected) >= max_columns:
            break
    return selected[:max_columns]


def plot_umap_from_obsm(adata_obj, study_id, color_by=None, basis="X_umap_seurat", point_size=1.0, alpha=0.75):
    if basis not in adata_obj.obsm:
        basis = "X_umap"
    coords = np.asarray(adata_obj.obsm[basis])[:, :2]
    fig, ax = plt.subplots(figsize=(6, 5))
    title = f"{study_id} {basis}"
    if color_by is None:
        ax.scatter(coords[:, 0], coords[:, 1], s=point_size, alpha=alpha, linewidths=0, rasterized=True)
    else:
        series = adata_obj.obs[color_by]
        title = f"{study_id} {basis} by {color_by}"
        if is_numeric_column(series):
            values = pd.to_numeric(series, errors="coerce").to_numpy()
            valid = np.isfinite(values)
            if valid.any():
                vmin = np.nanpercentile(values[valid], 1)
                vmax = np.nanpercentile(values[valid], 99)
            else:
                vmin = vmax = None
            scatter = ax.scatter(
                coords[valid, 0],
                coords[valid, 1],
                c=values[valid],
                s=point_size,
                alpha=alpha,
                linewidths=0,
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
                rasterized=True,
            )
            if (~valid).any():
                ax.scatter(coords[~valid, 0], coords[~valid, 1], s=point_size, alpha=0.2, linewidths=0, c="#d0d0d0", rasterized=True)
            fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.015, label=color_by)
        else:
            cats = pd.Categorical(series.astype("string").fillna("<NA>"))
            scatter = ax.scatter(
                coords[:, 0],
                coords[:, 1],
                c=cats.codes,
                s=point_size,
                alpha=alpha,
                linewidths=0,
                cmap="tab20",
                rasterized=True,
            )
            if len(cats.categories) <= 20:
                handles, _ = scatter.legend_elements(num=None)
                labels = list(cats.categories)
                ax.legend(handles[: len(labels)], labels, title=color_by, bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=4, fontsize=7)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal", adjustable="box")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    return fig, ax, basis


plot_manifest_rows = []
selected_color_rows = []
for study_id, adata_obj in adatas.items():
    study_plot_dir = UMAP_PLOT_DIR / study_id
    study_plot_dir.mkdir(parents=True, exist_ok=True)
    colors = select_umap_color_columns(adata_obj, max_columns=MAX_UMAP_COLOR_COLUMNS)
    for rank, column in enumerate(colors, start=1):
        selected_color_rows.append({"study_id": study_id, "rank": rank, "obs_column": column, "dtype": str(adata_obj.obs[column].dtype), "n_unique": int(adata_obj.obs[column].nunique(dropna=True))})

    for color_by in [None] + colors:
        fig, ax, basis = plot_umap_from_obsm(adata_obj, study_id, color_by=color_by)
        color_label = "uncolored" if color_by is None else sanitize_filename(color_by)
        out_path = study_plot_dir / f"umap_{basis}_{color_label}.png"
        if SAVE_PLOTS:
            fig.savefig(out_path, dpi=180, bbox_inches="tight")
        if SHOW_PLOTS:
            display(fig)
        plt.close(fig)
        plot_manifest_rows.append(
            {
                "study_id": study_id,
                "basis": basis,
                "color_by": "" if color_by is None else color_by,
                "path": str(out_path),
                "saved": SAVE_PLOTS and out_path.exists(),
            }
        )

umap_color_selection_df = pd.DataFrame(selected_color_rows)
umap_plot_manifest_df = pd.DataFrame(plot_manifest_rows)
umap_color_selection_df.to_csv(TABLE_DIR / "umap_color_column_selection.tsv", sep="\t", index=False)
umap_plot_manifest_df.to_csv(TABLE_DIR / "umap_plot_manifest.tsv", sep="\t", index=False)

display(umap_color_selection_df)
display(umap_plot_manifest_df)

## Final Output Manifest

In [ ]:
output_rows = []
for base in [TABLE_DIR, PLOT_DIR, LOG_DIR]:
    for path in sorted(base.rglob("*")):
        if path.is_file():
            output_rows.append(
                {
                    "kind": base.name,
                    "path": str(path),
                    "size_bytes": path.stat().st_size,
                }
            )
output_manifest_df = pd.DataFrame(output_rows)
output_manifest_df.to_csv(TABLE_DIR / "seurat_anndata_umap_inventory_output_manifest.tsv", sep="\t", index=False)

print("[SeuratAnnDataInventory] RUN_DIR", RUN_DIR)
print("[SeuratAnnDataInventory] tables", len(output_manifest_df.query('kind == "tables"')) if not output_manifest_df.empty else 0)
print("[SeuratAnnDataInventory] plots", len(output_manifest_df.query('kind == "plots"')) if not output_manifest_df.empty else 0)
print("[SeuratAnnDataInventory] logs", len(output_manifest_df.query('kind == "logs"')) if not output_manifest_df.empty else 0)
print("[SeuratAnnDataInventory] complete")

display(output_manifest_df.head(50))